In [1]:
import sys, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from greeks import delta, vanna, volga, charm

os.makedirs('../figures', exist_ok=True)
sns.set_theme(style='darkgrid')

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
S_range = np.linspace(50, 150, 300)

print(f'Setup OK  —  paramètres : S={S0}, K={K}, T={T}, r={r}, sigma={sigma}')


Setup OK  —  paramètres : S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2


# Greeks de second ordre : Vanna, Volga, Charm

Les Greeks de premier ordre (Delta, Gamma, Vega, Theta, Rho) mesurent la sensibilité du
**prix** aux paramètres de marché. Les Greeks de second ordre mesurent la sensibilité de
ces sensibilités — autrement dit, comment le **hedge lui-même** évolue quand les conditions
changent.

| Greek | Définition | Hedge concerné |
|-------|-----------|---------------|
| **Vanna** | ∂Delta/∂σ = ∂Vega/∂S | Delta hedge se déforme quand la vol bouge |
| **Volga** | ∂Vega/∂σ | Vega hedge se déforme quand la vol bouge |
| **Charm** | −∂Delta/∂T | Delta hedge change avec le seul passage du temps |


## 1. Vanna = ∂Delta/∂σ

**Formule** : $\text{Vanna} = -n(d_1) d_2 / \sigma$

**Intuition trader** : La Vanna mesure à quel point le **delta change quand la volatilité
implicite bouge**. Supposons qu'un market maker ait couvert son delta (delta-neutre).
Si la vol monte de 1 pt, son delta bouge de Vanna × 1 — il doit donc re-couvrir, même sans
mouvement du spot.

- **Vanna < 0 pour un call entre OTM et ATM** : quand la vol monte, le delta call diminue
  (l'option perd en directionnalité).
- **Vanna > 0 pour un call entre ATM et ITM** : quand la vol monte, le delta call augmente.
- **Vanna = 0 exactement ATM et deep ITM/OTM** : là où $d_2 = 0$ ou $n(d_1) \to 0$.

C'est aussi la sensibilité croisée $\partial\mathcal{V}/\partial S$ : si le spot monte,
le Vega change de Vanna × dS. Un long Vanna veut un marché qui monte ET vola qui monte
en même temps (mouvement corrélé spot-vol, typique d'une position en calls OTM).

In [2]:
vanna_call = np.array([vanna(S, K, T, r, sigma) for S in S_range])
vanna_put  = vanna_call   # identique call/put

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(S_range, vanna_call, lw=2, color='steelblue', label='Vanna (call = put)')
ax.axhline(0,   color='grey', lw=0.8, ls='--')
ax.axvline(K,   color='tomato', lw=1,  ls=':', label=f'ATM (K={K})')

# d2 = 0 ⟹ Vanna = 0 ⟹ S ≈ K·exp(-(r - σ²/2)·T)
S_zero = K * np.exp(-(r - 0.5*sigma**2) * T)

ax.annotate('Vanna > 0\n(delta ↑ si vol ↑)',
            xy=(80, vanna_call[np.argmin(np.abs(S_range - 80))]),
            xytext=(55, 1.0), fontsize=9, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue'))
ax.annotate('Vanna < 0\n(delta ↓ si vol ↑)',
            xy=(118, vanna_call[np.argmin(np.abs(S_range - 118))]),
            xytext=(122, -0.6), fontsize=9, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue'))
ax.annotate(f'Vanna = 0\n($d_2$ = 0, S ≈ {S_zero:.0f})',
            xy=(S_zero, 0), xytext=(103, 0.3), fontsize=9, color='tomato',
            arrowprops=dict(arrowstyle='->', color='tomato'))

ax.set_title('Vanna vs Spot — sensibilité du Delta à la volatilité', fontsize=13)
ax.set_xlabel('Spot S')
ax.set_ylabel('Vanna = ∂Delta/∂σ')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/11_vanna_vs_spot.png', dpi=150)
plt.close(fig)
print('Saved: figures/11_vanna_vs_spot.png')

Saved: figures/11_vanna_vs_spot.png


## 2. Volga = ∂Vega/∂σ  (aussi appelé Vomma)

**Formule** : $\text{Volga} = S n(d_1)\sqrt{T} \cdot d_1 d_2 / \sigma$

**Intuition trader** : Le Volga mesure la **convexité par rapport à la vol** : combien le
Vega change quand la vol bouge elle-même. C'est l'analogue du Gamma (convexité en spot)
mais dans la dimension volatilité.

- **Volga ≈ 0 ATM** : à la monnaie, $d_1 \approx d_2 \approx 0$, donc le produit $d_1 d_2 \approx 0$.
  Le Vega d'une option ATM ne change presque pas avec la vol.
- **Volga > 0 deep OTM et deep ITM** : le produit $d_1 d_2 > 0$ sur les ailes (même signe).
  Les options OTM deviennent exponentiellement plus sensibles à la vol.

**Lien avec le smile** : un portefeuille long Volga (long options OTM) bénéficie d'une hausse
de vol, parce que ses options OTM gagnent plus de Vega qu'elles n'en perdent. C'est pourquoi
les options OTM se paient plus cher qu'une vol constante ne le prédirait — le marché intègre
implicitement le coût de couverture du Volga dans leurs prix, produisant le **smile**.

In [3]:
volga_vals = np.array([volga(S, K, T, r, sigma) for S in S_range])

# Minimum de la courbe (transition entre les deux bosses, d1·d2 le plus négatif)
idx_min = np.argmin(volga_vals)
S_min   = S_range[idx_min]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(S_range, volga_vals, lw=2, color='darkorange', label='Volga (= Vomma)')
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.axvline(K, color='tomato', lw=1, ls=':', label=f'ATM (K={K})')
ax.fill_between(S_range, volga_vals, 0,
                where=(volga_vals > 0), alpha=0.15, color='darkorange',
                label='Volga > 0 (ailes OTM/ITM)')

ax.annotate(f'Volga faible\nautour de l\'ATM\n(S ≈ {S_min:.0f})',
            xy=(S_min, volga_vals[idx_min]),
            xytext=(80, -4), fontsize=9, color='tomato',
            arrowprops=dict(arrowstyle='->', color='tomato'))
ax.annotate('Bosse OTM\n(d1·d2 > 0)',
            xy=(75, volga_vals[np.argmin(np.abs(S_range - 75))]),
            xytext=(55, 10), fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.annotate('Bosse ITM\n(d1·d2 > 0)',
            xy=(128, volga_vals[np.argmin(np.abs(S_range - 128))]),
            xytext=(130, 10), fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))

ax.set_title('Volga vs Spot — convexité en volatilité et lien avec le smile', fontsize=13)
ax.set_xlabel('Spot S')
ax.set_ylabel('Volga = ∂Vega_brut/∂σ')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/12_volga_vs_spot.png', dpi=150)
plt.close(fig)
print('Saved: figures/12_volga_vs_spot.png')

Saved: figures/12_volga_vs_spot.png


## 3. Charm = −∂Delta/∂T  (delta bleed)

**Formule** : $\text{Charm} = -n(d_1)\cdot(2rT - d_2 \sigma\sqrt{T}) / (2T\sigma\sqrt{T})$

Retourné en unités annuelles ; diviser par 365 pour obtenir la variation du delta par jour
calendaire (analogue au Theta pour le prix).

**Intuition trader** : Le Charm représente le **rééquilibrage de delta imposé par le seul
passage du temps**, sans mouvement du spot ni de la vol. Un market maker delta-neutre en
fin de semaine doit ajuster sa couverture le lundi matin, même si le marché n'a pas bougé.
C'est le "**delta bleed**".

- **Charm négatif sur les calls ATM/légèrement ITM** : le delta call glisse vers 0.5 à
  l'approche de l'expiry — la valeur temps s'évapore, réduisant l'exposition directionnelle.
- **Charm positif sur les calls deep ITM** : le delta call converge vers 1, donc augmente.
- **Charm des puts** : symétrique, analogue avec convergence vers −1 ou 0.

*Exemple concret* : un trader est delta-neutre le vendredi soir sur un call légèrement OTM.
Le lundi matin, sans aucune actualité, son delta a bougé de Charm × (3/365) — il doit
racheter des actions pour se recouvrir, ce qui génère un P&L négatif lié au temps.

In [4]:
charm_call = np.array([charm(S, K, T, r, sigma, 'call') for S in S_range])
charm_put  = np.array([charm(S, K, T, r, sigma, 'put')  for S in S_range])

# En variation par jour (÷ 365)
charm_call_day = charm_call / 365
charm_put_day  = charm_put  / 365

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(S_range, charm_call_day * 1000, lw=2, color='steelblue',
        label='Charm call (×1000, /jour)')
ax.plot(S_range, charm_put_day  * 1000, lw=2, color='darkorange', ls='--',
        label='Charm put (×1000, /jour)')
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.axvline(K, color='tomato', lw=1, ls=':', label=f'ATM (K={K})')

ax.annotate('Call ITM : delta bleed\nvers 1 (Charm > 0)',
            xy=(130, charm_call_day[np.argmin(np.abs(S_range-130))]*1000),
            xytext=(133, 0.05), fontsize=9, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue'))
ax.annotate('Call OTM : delta bleed\nvers 0 (Charm < 0)',
            xy=(72, charm_call_day[np.argmin(np.abs(S_range-72))]*1000),
            xytext=(52, -0.08), fontsize=9, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue'))

ax.set_title('Charm vs Spot — delta bleed par jour (×1000)', fontsize=13)
ax.set_xlabel('Spot S')
ax.set_ylabel('Charm / 365  ×1000  (variation Δ par jour)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/13_charm_vs_spot.png', dpi=150)
plt.close(fig)
print('Saved: figures/13_charm_vs_spot.png')

Saved: figures/13_charm_vs_spot.png


### Charm vs Maturité — accélération du delta bleed en fin de vie

Comme le Theta, le Charm s'intensifie à l'approche de l'expiry :
les rééquilibrages de delta deviennent plus fréquents et plus importants
quand le temps restant est faible.


In [5]:
T_range = np.linspace(0.02, 2.0, 300)   # de ~1 semaine à 2 ans

# ATM (S = K = 100) et légèrement OTM (S = 90)
charm_atm_ann  = np.array([charm(S0, K, t, r, sigma, 'call')         for t in T_range])
charm_otm_ann  = np.array([charm(90,  K, t, r, sigma, 'call')         for t in T_range])
charm_itm_ann  = np.array([charm(110, K, t, r, sigma, 'call')         for t in T_range])

charm_atm_day  = charm_atm_ann / 365
charm_otm_day  = charm_otm_ann / 365
charm_itm_day  = charm_itm_ann / 365

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(T_range, charm_atm_day * 1000, lw=2, color='steelblue',
        label='ATM  (S=100)')
ax.plot(T_range, charm_otm_day * 1000, lw=2, color='tomato', ls='--',
        label='OTM  (S=90)')
ax.plot(T_range, charm_itm_day * 1000, lw=2, color='seagreen', ls=':',
        label='ITM  (S=110)')
ax.axhline(0, color='grey', lw=0.8, ls='--')

ax.set_title('Charm vs Maturité — accélération du delta bleed en fin de vie', fontsize=13)
ax.set_xlabel('Maturité T (années)')
ax.set_ylabel('Charm / 365  ×1000  (variation Δ par jour)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/14_charm_vs_maturity.png', dpi=150)
plt.close(fig)
print('Saved: figures/14_charm_vs_maturity.png')

Saved: figures/14_charm_vs_maturity.png


## Synthèse — ce que les Greeks de second ordre apportent au trader

| Greek | Situation où il importe | Risque si ignoré |
|-------|------------------------|-----------------|
| **Vanna** | Vol qui bouge fortement (annonce macro, event) | Delta hedge obsolète dès que la vol bouge |
| **Volga** | Long/short vol sur les ailes, structuration de produits exotiques | Sous-estimation du coût de couverture du Vega ; smile mal reproduit |
| **Charm** | Positions tenues sur le week-end, options courtes maturités | Exposition directionnelle non anticipée au réouverture |

Les trois Greeks sont **nuls dans un monde Black-Scholes à vol constante** uniquement si
on rebalance en continu. En pratique, le rebalancement est discret (journalier, hebdomadaire)
et toute variation de vol ou de temps génère un P&L dépendant de Vanna, Volga et Charm.
C'est pourquoi les books d'options industriels trackent ces risques en temps réel.


In [6]:
# Valeurs numériques de référence ATM
print(f'Paramètres ATM : S={S0}, K={K}, T={T}, r={r}, sigma={sigma}')
print()
print(f'  Vanna  = {vanna(S0,K,T,r,sigma):+.6f}')
print(f'           interp. : si vol +1% → delta change de {vanna(S0,K,T,r,sigma)*0.01:+.4f}')
print()
print(f'  Volga  = {volga(S0,K,T,r,sigma):+.6f}  (en Vega_brut par unité de vol)')
print(f'           interp. : si vol +1% → Vega_brut change de {volga(S0,K,T,r,sigma)*0.01:+.4f}')
print()
charm_day = charm(S0,K,T,r,sigma,'call') / 365
print(f'  Charm  = {charm(S0,K,T,r,sigma,"call"):+.6f} /an  →  {charm_day:+.6f} /jour')
print(f'           interp. : delta bleed sur 3 jours = {charm_day*3:+.4f}')
print()
print('Fichiers PNG créés :')
for f in ['11_vanna_vs_spot.png','12_volga_vs_spot.png',
          '13_charm_vs_spot.png','14_charm_vs_maturity.png']:
    import os
    path = f'../figures/{f}'
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'  figures/{f}  ({size:,} bytes)')


Paramètres ATM : S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2

  Vanna  = -0.281430
           interp. : si vol +1% → delta change de -0.0028

  Volga  = +9.850059  (en Vega_brut par unité de vol)
           interp. : si vol +1% → Vega_brut change de +0.0985

  Charm  = -0.065667 /an  →  -0.000180 /jour
           interp. : delta bleed sur 3 jours = -0.0005

Fichiers PNG créés :
  figures/11_vanna_vs_spot.png  (86,348 bytes)
  figures/12_volga_vs_spot.png  (110,997 bytes)
  figures/13_charm_vs_spot.png  (103,133 bytes)
  figures/14_charm_vs_maturity.png  (82,171 bytes)
